# 03 — Rates Regime & Yield Curve Velocity

Detect regime changes in the yield curve using spectral velocity.
Requires free FRED API key: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'specvel'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

FRED_KEY = os.environ.get('FRED_API_KEY', 'your_key_here')

if FRED_KEY == 'your_key_here':
    raise ValueError('Set FRED_KEY. Free at: https://fred.stlouisfed.org/docs/api/api_key.html')

from adapters.fixed_income import FixedIncomeAdapter
from core import compute_velocity
from anomaly import detect_anomaly
from leaderboard import run_leaderboard, print_leaderboard, save_leaderboard

START = '2018-01-01'
END   = '2026-03-10'

In [ ]:
adapter = FixedIncomeAdapter(api_key=FRED_KEY)
df = run_leaderboard(adapter, start=START, end=END,
                     asset_class='fixed_income', top_n=20)
print_leaderboard(df, title='FIXED INCOME VELOCITY LEADERBOARD')

## Yield Curve — 2Y vs 10Y vs Spread Velocity

In [ ]:
curve_series = {'DGS2': '2Y', 'DGS10': '10Y', 'T10Y2Y': '10Y-2Y Spread'}

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

for ax, (sid, label) in zip(axes, curve_series.items()):
    try:
        raw    = adapter.fetch(sid, START, END)
        normed = adapter.normalize(raw)
        vel    = compute_velocity(normed)
        anom   = detect_anomaly(normed)

        ax2 = ax.twinx()
        ax.plot(raw.index, raw.values, color='steelblue', lw=1.5, label=label)
        ax2.plot(vel.index, vel.values, color='darkorange', lw=1.2, alpha=0.8,
                 label='Velocity')
        ax2.axhline(0, color='black', lw=0.5, ls='--')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values > 0), alpha=0.12, color='green')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values < 0), alpha=0.12, color='red')

        flag = '⚠ ANOMALY' if anom['flag'] else ''
        ax.set_ylabel(f'{label} (%)', fontsize=9)
        ax2.set_ylabel('Velocity', fontsize=9, color='darkorange')
        ax.set_title(f'{label}  {flag}', fontsize=10, fontweight='bold')
        ax.legend(loc='upper left', fontsize=8)
    except Exception as e:
        ax.set_title(f'{sid}: {e}')

plt.suptitle('Yield Curve Spectral Velocity', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/yield_curve_velocity.png', dpi=150)
plt.show()

In [ ]:
save_leaderboard(df, '../results/fixed_income_leaderboard.csv')
print('Saved.')